# Agent Evaluation: MLflow GenAI Scorers and LLM-as-Judge

This notebook evaluates the Custom Deep Research pipeline using MLflow's GenAI
evaluation framework. It covers:

- **Prompt Registry**: Register versioned evaluation prompts, link traces to prompt versions
- **GenAI Datasets**: Create persistent, versioned test datasets on the MLflow server
- **Custom Scorers**: Deterministic checks (fast, no LLM calls)
- **LLM-as-Judge**: Built-in scorers for relevance, safety, and guideline compliance
- **Two-mode evaluation**: Simple (fast) vs LLM-Judge (comprehensive)

## 1. Load Environment

In [ ]:
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "")
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "deep-research-harness")
llm_base_url = os.getenv("LLM_BASE_URL", "")
llm_model = os.getenv("LLM_MODEL", "")
llm_api_key = os.getenv("LLM_API_KEY", "") or os.getenv("MAAS_API_KEY", "")

print(f"MLflow Tracking URI: {mlflow_uri or '(not set)'}")
print(f"Experiment: {experiment_name}")
print(f"LLM Base URL: {llm_base_url or '(not set)'}")
print(f"LLM Model: {llm_model or '(not set)'}")

missing = []
if not mlflow_uri:
    missing.append("MLFLOW_TRACKING_URI")
if missing:
    print(f"\u26a0\ufe0f Missing: {', '.join(missing)} \u2014 configure in .env")
else:
    print("\u2705 Environment loaded")

## 2. Configure MLflow

In [ ]:
import logging
import mlflow

logging.getLogger("mlflow").setLevel(logging.ERROR)

mlflow.set_tracking_uri(mlflow_uri)

eval_experiment = experiment_name
if not eval_experiment.endswith("-eval"):
    eval_experiment = f"{eval_experiment}-eval"

mlflow.set_experiment(eval_experiment)
exp = mlflow.get_experiment_by_name(eval_experiment)

try:
    mlflow.langchain.autolog()
except Exception:
    pass

print(f"Experiment: {eval_experiment}")
if exp:
    print(f"Experiment ID: {exp.experiment_id}")
print("\u2705 MLflow configured")

## 3. Register Evaluation Prompt in MLflow Prompt Registry

Register the research quality evaluation prompt as a versioned artifact. This
enables prompt-to-trace linkage: every evaluation trace is automatically associated
with the prompt version that produced it.

In [ ]:
prompt_name = "research-quality-judge"

quality_prompt_template = """You are a research quality evaluator. Score the following
research report on a scale of 1-10 across these dimensions:
- Relevance: Does the report address the query?
- Completeness: Are all aspects of the query covered?
- Accuracy: Are claims supported by citations?
- Coherence: Is the report well-structured and readable?

Query: {{query}}
Report: {{report}}

Return a JSON object with scores for each dimension and an overall score:
{"relevance": <1-10>, "completeness": <1-10>, "accuracy": <1-10>,
 "coherence": <1-10>, "overall": <1-10>, "reasoning": "<text>"}"""

try:
    registered_prompt = mlflow.genai.register_prompt(
        name=prompt_name,
        template=quality_prompt_template,
        commit_message="Research quality judge prompt for deep research evaluation",
        tags={
            "type": "evaluation-judge",
            "domain": "deep-research",
        },
    )
    print(f"\u2705 Registered prompt: {prompt_name} v{registered_prompt.version}")
except Exception as e:
    if "already exists" in str(e).lower() or "RESOURCE_ALREADY_EXISTS" in str(e):
        registered_prompt = mlflow.genai.register_prompt(
            name=prompt_name,
            template=quality_prompt_template,
            commit_message="Updated research quality judge prompt",
            tags={"type": "evaluation-judge", "domain": "deep-research"},
        )
        print(f"\u2705 Updated prompt: {prompt_name} v{registered_prompt.version}")
    else:
        raise

prompt_uri = f"prompts:/{prompt_name}/{registered_prompt.version}"
print(f"Prompt URI: {prompt_uri}")

## 4. Create Evaluation Dataset

Create a persistent dataset on the MLflow server using `mlflow.genai.datasets`.
Each test case has:
- `inputs`: The research query
- `expectations`: Expected behavior (keywords, topics, constraints)

In [ ]:
from mlflow.genai.datasets import create_dataset

test_cases = [
    {
        "inputs": {"query": "What is the main architecture described in the document?"},
        "expectations": {
            "expected_keywords": ["architecture", "system", "design"],
            "min_length": 200,
            "requires_citations": True,
        },
    },
    {
        "inputs": {"query": "What are the key contributions and innovations?"},
        "expectations": {
            "expected_keywords": ["contribution", "novel"],
            "min_length": 200,
            "requires_citations": True,
        },
    },
    {
        "inputs": {"query": "How does the system handle document parsing and conversion?"},
        "expectations": {
            "expected_keywords": ["parsing", "document", "convert"],
            "min_length": 150,
            "requires_citations": True,
        },
    },
    {
        "inputs": {"query": "What evaluation metrics are used and what are the results?"},
        "expectations": {
            "expected_keywords": ["metric", "evaluation", "result"],
            "min_length": 200,
            "requires_citations": True,
        },
    },
    {
        "inputs": {"query": "What are the limitations and future work directions?"},
        "expectations": {
            "expected_keywords": ["limitation", "future"],
            "min_length": 150,
            "requires_citations": False,
        },
    },
]

dataset_name = "deep_research_eval"
dataset = create_dataset(
    name=dataset_name,
    tags={"stage": "validation", "version": "1", "domain": "deep-research"},
)
dataset = dataset.merge_records(test_cases)

print(f"\u2705 Dataset created: {dataset.dataset_id}")
print(f"Test cases: {len(test_cases)}")
for i, tc in enumerate(test_cases, 1):
    print(f"  {i}. {tc['inputs']['query']}")

## 5. Define Custom Deterministic Scorers

These scorers run without LLM calls, making them fast for regression testing.
They check structural properties of the research report.

In [ ]:
import re
from mlflow.genai.scorers import scorer


@scorer
def contains_expected_keywords(inputs: dict, outputs: str, expectations: dict) -> bool:
    """Check if the report contains expected topic keywords."""
    keywords = expectations.get("expected_keywords", [])
    if not keywords:
        return True
    text_lower = str(outputs).lower()
    return any(kw.lower() in text_lower for kw in keywords)


@scorer
def has_citations(outputs: str, expectations: dict) -> bool:
    """Check if the report includes source references when required."""
    if not expectations.get("requires_citations", False):
        return True
    citation_patterns = [
        r"\[\d+\]",
        r"\(.*?\d{4}.*?\)",
        r"source:",
        r"according to",
        r"cited in",
        r"reference",
    ]
    text = str(outputs).lower()
    return any(re.search(p, text, re.IGNORECASE) for p in citation_patterns)


@scorer
def response_depth(outputs: str, expectations: dict) -> float:
    """Score response depth based on length threshold."""
    min_length = expectations.get("min_length", 100)
    length = len(str(outputs))
    if length >= min_length:
        return 1.0
    elif length >= min_length * 0.5:
        return 0.5
    return 0.0


simple_scorers = [contains_expected_keywords, has_citations, response_depth]

print("Simple scorers (no LLM calls):")
for s in simple_scorers:
    print(f"  - {s.name}")
print("\u2705 Scorers defined")

## 6. Define LLM-as-Judge Scorers

These scorers use an LLM to evaluate response quality across multiple dimensions.
They require `LLM_BASE_URL` and `LLM_MODEL` to be configured.

In [ ]:
from mlflow.genai.scorers import Guidelines, RelevanceToQuery, Safety

llm_scorers = []

if llm_base_url and llm_model:
    os.environ["OPENAI_API_BASE"] = llm_base_url
    os.environ["OPENAI_BASE_URL"] = llm_base_url
    if llm_api_key:
        os.environ["OPENAI_API_KEY"] = llm_api_key

    judge_model = f"openai:/{llm_model}"
    print(f"Judge model: {judge_model}")

    llm_scorers = [
        RelevanceToQuery(model=judge_model),
        Safety(model=judge_model),
        Guidelines(
            name="research_report_guidelines",
            guidelines=[
                "The response should directly address the research query",
                "The response should cite or reference source material",
                "The response should be well-structured with clear sections",
                "The response should NOT hallucinate unsupported claims",
            ],
            model=judge_model,
        ),
    ]
    print(f"\u2705 LLM judge scorers: {len(llm_scorers)}")
else:
    print("\u26a0\ufe0f LLM judge not configured \u2014 set LLM_BASE_URL and LLM_MODEL in .env")
    print("  Simple evaluation will still work")

## 7. Define Predictor

The predictor wraps the research pipeline invocation for `mlflow.genai.evaluate()`.
It sends a query to the backend and returns the research report.

In [ ]:
import time
import httpx

BACKEND_URL = f"http://localhost:{os.getenv('BACKEND_PORT', '8000')}"


@mlflow.trace
def predict_fn(query: str) -> str:
    """Execute a research query and return the report text."""
    try:
        if prompt_uri:
            mlflow.genai.load_prompt(prompt_uri)
    except Exception:
        pass

    try:
        resp = httpx.post(
            f"{BACKEND_URL}/research",
            json={"query": query},
            timeout=180,
        )
        if resp.status_code == 200:
            data = resp.json()
            return data.get("report", data.get("result", str(data)))
        return f"Error: HTTP {resp.status_code}"
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


print(f"Backend URL: {BACKEND_URL}")
print(f"Prompt linkage: {prompt_uri}")
print("\u2705 Predictor defined")

## 8. Run Simple Evaluation (Fast, No LLM)

Run deterministic scorers only. This is useful for quick regression checks.

In [ ]:
print("Running simple evaluation...")
print(f"Dataset: {len(test_cases)} examples")
print(f"Scorers: {len(simple_scorers)} (deterministic)")
print(f"Prompt: {prompt_uri}")
print()

simple_results = mlflow.genai.evaluate(
    data=dataset,
    predict_fn=predict_fn,
    scorers=simple_scorers,
)

print("\nSimple Evaluation Results:")
print("-" * 40)
for metric, value in sorted(simple_results.metrics.items()):
    if isinstance(value, float):
        print(f"  {metric}: {value:.2%}")
    else:
        print(f"  {metric}: {value}")
print("\n\u2705 Simple evaluation complete")

## 9. Run LLM-as-Judge Evaluation (Comprehensive)

Full evaluation with both deterministic and LLM-based scorers. This takes longer
but provides deeper quality assessment.

In [ ]:
if llm_scorers:
    all_scorers = simple_scorers + llm_scorers

    print("Running LLM-as-Judge evaluation...")
    print(f"Dataset: {len(test_cases)} examples")
    print(f"Scorers: {len(all_scorers)} ({len(llm_scorers)} LLM judges)")
    print(f"Prompt: {prompt_uri}")
    print()

    full_results = mlflow.genai.evaluate(
        data=dataset,
        predict_fn=predict_fn,
        scorers=all_scorers,
    )

    print("\nFull Evaluation Results:")
    print("-" * 40)
    for metric, value in sorted(full_results.metrics.items()):
        if isinstance(value, float):
            print(f"  {metric}: {value:.2%}")
        else:
            print(f"  {metric}: {value}")
    print("\n\u2705 LLM-as-Judge evaluation complete")
else:
    print("\u26a0\ufe0f Skipping LLM-as-Judge evaluation (not configured)")
    print("Set LLM_BASE_URL and LLM_MODEL in .env for full evaluation")

## 10. View Results in MLflow UI

Navigate to the MLflow UI to see detailed per-trace assessments:

1. Open your MLflow tracking URI in a browser
2. Go to **Experiments** > `deep-research-harness-eval`
3. Click on **Evaluation Runs** tab
4. Select your run to view per-trace assessments
5. Enable **All Assessments** in the Columns dropdown to see LLM judge scores
6. Click on a trace to see the linked prompt version under the **Prompt** tab

View registered prompts under **Prompts** in the left sidebar.
View datasets under **Experiments** > Default > **Datasets**.

In [ ]:
print(f"View evaluation results: {mlflow_uri}/#/experiments")
print(f"View registered prompts: {mlflow_uri}/#/prompts")
print(f"Dataset ID: {dataset.dataset_id}")
print(f"Prompt: {prompt_uri}")
print("\u2705 Check the MLflow UI for detailed results")

## 11. Summary

In [ ]:
print("\u2705 Agent evaluation walkthrough complete")
print()
print("What we covered:")
print("  1. Registered versioned evaluation prompt in MLflow Prompt Registry")
print("  2. Created persistent evaluation dataset with mlflow.genai.datasets")
print("  3. Defined custom deterministic scorers (keywords, citations, depth)")
print("  4. Defined LLM-as-Judge scorers (relevance, safety, guidelines)")
print("  5. Ran simple evaluation (fast, no LLM)")
print("  6. Ran LLM-as-Judge evaluation (comprehensive)")
print("  7. Viewed results in MLflow UI")
print()
print("Architecture:")
print("  Test Dataset  \u2192  Research Pipeline  \u2192  Scorers  \u2192  MLflow Experiment")
print("  (GenAI Dataset)  (backend /research)   (simple    (metrics + traces)")
print("                                          + judge)")
print()
print("Next: 5_evaluation_pipeline.ipynb \u2014 Run evaluation as a KFP pipeline on RHOAI")